# Review Generated Schema.org JSON-LD

Inspect the Gemini 2.5 Flash generation output before moving to training data prep.

Checks:
1. Overall valid/invalid/failed breakdown
2. Quality score distribution
3. Type distribution (generated vs hint)
4. Property count stats
5. Random samples per type for manual review
6. Common failure modes

In [ ]:
import json, glob, random, os
from pathlib import Path
from collections import Counter
from IPython.display import display, HTML, JSON

GEN_DIR = Path('../data/generated')
files = sorted(glob.glob(str(GEN_DIR / '*.json')))
print(f'Total generated files: {len(files):,}')

In [ ]:
# Load all results
results = []
for f in files:
    with open(f) as fh:
        results.append(json.load(fh))

valid   = [r for r in results if r['valid']]
invalid = [r for r in results if not r['valid']]

print(f'Valid:   {len(valid):,} ({len(valid)/len(results)*100:.1f}%)')
print(f'Invalid: {len(invalid):,} ({len(invalid)/len(results)*100:.1f}%)')

## Quality Score Distribution

In [ ]:
scores = [r['quality_score'] for r in valid]

# Histogram buckets
buckets = Counter()
for s in scores:
    bucket = round(s, 1)
    buckets[bucket] += 1

print(f'Quality score stats:')
print(f'  Min:    {min(scores):.3f}')
print(f'  Max:    {max(scores):.3f}')
print(f'  Mean:   {sum(scores)/len(scores):.3f}')
print(f'  Median: {sorted(scores)[len(scores)//2]:.3f}')
print()
print('Distribution:')
for bucket in sorted(buckets.keys()):
    bar = '#' * (buckets[bucket] // 20)
    print(f'  {bucket:.1f}  {buckets[bucket]:5,}  {bar}')

## Type Distribution

In [ ]:
# What types did Gemini actually generate vs what we hinted?
hint_types = Counter(r['schema_type_hint'] for r in valid)
gen_types  = Counter(r['schema_types'][0] for r in valid if r['schema_types'])

all_types = sorted(set(hint_types.keys()) | set(gen_types.keys()), 
                   key=lambda t: -gen_types.get(t, 0))

print(f'{"Generated Type":35s} {"Count":>6s}   {"Hint Type":35s} {"Count":>6s}')
print('-' * 90)
for i, t in enumerate(all_types[:25]):
    gen_str  = f'{t:35s} {gen_types.get(t, 0):6,}'
    hint_str = ''
    if i < len(hint_types.most_common(25)):
        ht, hc = hint_types.most_common(25)[i]
        hint_str = f'{ht:35s} {hc:6,}'
    print(f'{gen_str}   {hint_str}')

print(f'\nTotal unique generated types: {len(gen_types)}')
print(f'Total unique hint types:     {len(hint_types)}')

## Property Count Stats

In [ ]:
props = [r['property_count'] for r in valid]

print(f'Property count stats (valid only):')
print(f'  Min:    {min(props)}')
print(f'  Max:    {max(props)}')
print(f'  Mean:   {sum(props)/len(props):.1f}')
print(f'  Median: {sorted(props)[len(props)//2]}')
print()

# By type
props_by_type = {}
for r in valid:
    t = r['schema_types'][0] if r['schema_types'] else '?'
    props_by_type.setdefault(t, []).append(r['property_count'])

print(f'{"Type":30s} {"Count":>6s} {"Avg Props":>10s} {"Min":>5s} {"Max":>5s}')
print('-' * 60)
for t in sorted(props_by_type, key=lambda x: -len(props_by_type[x]))[:20]:
    p = props_by_type[t]
    print(f'{t:30s} {len(p):6,} {sum(p)/len(p):10.1f} {min(p):5d} {max(p):5d}')

## Visual Review — Screenshot + HTML + Generated Schema

Pick a random page (re-run cell to get a new one), see the screenshot the model saw, the HTML snippet, and what Gemini generated.

In [ ]:
import hashlib, base64, re
from IPython.display import display, HTML, Image

# Load the source JSONL for HTML content
pages_by_url = {}
with open('../data/processed/pages_for_generation.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        if rec['url'] not in pages_by_url:
            pages_by_url[rec['url']] = rec

SHOT_DIR = Path('../data/screenshots_v2')

def show_page(idx=None):
    """Show a single page: screenshot + HTML preview + generated schema."""
    pool = valid if valid else results
    if idx is None:
        r = random.choice(pool)
    else:
        r = pool[idx % len(pool)]
    
    url = r['url']
    h = hashlib.md5(url.encode()).hexdigest()
    shot_path = SHOT_DIR / f'{h}.png'
    source = pages_by_url.get(url, {})
    html_raw = source.get('html', '(not found in JSONL)')
    
    # Quality badge color
    qs = r['quality_score']
    if qs >= 0.6: badge_color = '#2ecc71'
    elif qs >= 0.4: badge_color = '#f39c12'
    else: badge_color = '#e74c3c'
    
    # Build display
    header_html = f"""
    <div style="border:2px solid #333; border-radius:8px; padding:16px; margin:8px 0; background:#1a1a2e; color:#eee; font-family:monospace;">
        <h3 style="margin:0 0 8px 0; color:#fff;">Page Review</h3>
        <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:center; margin-bottom:12px;">
            <span style="background:{badge_color}; color:#fff; padding:4px 12px; border-radius:12px; font-weight:bold;">
                Quality: {qs:.2f}
            </span>
            <span style="background:#3498db; color:#fff; padding:4px 12px; border-radius:12px;">
                {r['schema_types'][0] if r['schema_types'] else '?'}
            </span>
            <span style="background:#555; color:#fff; padding:4px 12px; border-radius:12px;">
                {r['property_count']} properties
            </span>
            <span style="background:#555; color:#fff; padding:4px 12px; border-radius:12px;">
                Hint: {r.get('schema_type_hint', '?')}
            </span>
        </div>
        <div style="color:#aaa; word-break:break-all; font-size:12px;">
            <a href="{url}" style="color:#6cb4ee;" target="_blank">{url}</a>
        </div>
    </div>
    """
    display(HTML(header_html))
    
    # Screenshot
    if shot_path.exists():
        display(HTML('<h4 style="color:#555;">Screenshot (what the model saw)</h4>'))
        display(Image(filename=str(shot_path), width=700))
    else:
        display(HTML('<p style="color:red;">Screenshot not found</p>'))
    
    # HTML preview (first 3000 chars, escaped)
    html_preview = html_raw[:3000]
    # Extract title
    title_match = re.search(r'<title[^>]*>([^<]+)</title>', html_raw, re.IGNORECASE)
    title = title_match.group(1).strip()[:100] if title_match else '(no title)'
    
    html_escaped = html_preview.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    display(HTML(f"""
    <details style="margin:8px 0;">
        <summary style="cursor:pointer; font-weight:bold; color:#555;">
            HTML Source — "{title}" ({len(html_raw):,} chars total, showing first 3K)
        </summary>
        <pre style="background:#f5f5f5; padding:12px; border-radius:4px; overflow-x:auto; 
                    font-size:11px; max-height:400px; overflow-y:auto; white-space:pre-wrap;">{html_escaped}{'...' if len(html_raw) > 3000 else ''}</pre>
    </details>
    """))
    
    # Generated JSON-LD
    try:
        parsed_jsonld = json.loads(r['generated_jsonld'])
        formatted = json.dumps(parsed_jsonld, indent=2, ensure_ascii=False)
    except:
        formatted = r['generated_jsonld']
    
    jsonld_escaped = formatted.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    display(HTML(f"""
    <details open style="margin:8px 0;">
        <summary style="cursor:pointer; font-weight:bold; color:#555;">
            Generated JSON-LD ({len(formatted):,} chars)
        </summary>
        <pre style="background:#1e1e1e; color:#d4d4d4; padding:12px; border-radius:4px; 
                    overflow-x:auto; font-size:12px; max-height:600px; overflow-y:auto;">{jsonld_escaped}</pre>
    </details>
    """))

# Show one random page
show_page()

In [ ]:
# Re-run this cell to see another random page, or pass an index:
# show_page(42)      # specific page
# show_page()        # random
show_page()

## Failure Analysis

In [ ]:
error_types = Counter()
for r in invalid:
    for e in r.get('errors', []):
        # Normalize error messages
        if 'Unterminated string' in e:
            error_types['Truncated output (unterminated string)'] += 1
        elif 'Missing required' in e:
            error_types[e] += 1
        elif 'Invalid JSON' in e:
            error_types['Invalid JSON (other)'] += 1
        else:
            error_types[e[:60]] += 1

print(f'Error breakdown ({len(invalid)} invalid results):')
print()
for err, cnt in error_types.most_common():
    print(f'  {cnt:5,}  {err}')

In [ ]:
# Show a few invalid examples to understand patterns
print('Sample invalid outputs:\n')
for r in random.sample(invalid, min(5, len(invalid))):
    print(f'URL: {r["url"][:70]}')
    print(f'Hint: {r["schema_type_hint"]}  |  Errors: {r["errors"]}')
    print(f'Last 150 chars: ...{r["generated_jsonld"][-150:]}')
    print()

## Summary

In [ ]:
high_q = [r for r in valid if r['quality_score'] >= 0.4]

print('=' * 50)
print('GENERATION SUMMARY')
print('=' * 50)
print(f'Total generated:  {len(results):,}')
print(f'Valid:            {len(valid):,} ({len(valid)/len(results)*100:.1f}%)')
print(f'High quality:     {len(high_q):,} ({len(high_q)/len(results)*100:.1f}%)  (score >= 0.4)')
print(f'Invalid:          {len(invalid):,}')
print(f'Unique gen types: {len(gen_types)}')
print(f'Avg properties:   {sum(props)/len(props):.1f}')
print()
print('Ready for training data prep (notebook 05)?')
if len(high_q) >= 5000:
    print(f'  YES - {len(high_q):,} high-quality examples is sufficient.')
else:
    print(f'  MAYBE - {len(high_q):,} examples may be low. Consider re-running failed pages.')